# Smoke test: `analytics` hit-counter service

Two targets, toggled by `LOCAL` below:
- **`LOCAL = True`** — `npm run dev` running in another terminal (file storage,
  no credentials needed at all — `localhost:8085`).
- **`LOCAL = False`** — the deployed `https://analytics.fretchen.eu`, real S3
  (needs `SCW_ACCESS_KEY`/`SCW_SECRET_KEY` in `analytics/.env`, copied from
  `analytics/.env.example`, only for the read-back step — the `POST /hit` call
  itself needs no credentials either way).

In [ ]:
import os
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

from storage import LocalStorage, S3Storage

load_dotenv()  # searches upward — finds ../.env (analytics/.env)

LOCAL = True  # npm run dev (file storage) vs the deployed service

ANALYTICS_URL = "http://localhost:8085" if LOCAL else "https://analytics.fretchen.eu"

## 1. Valid hits — expect `204`

In [ ]:
for _ in range(2):
    response = requests.post(
        f"{ANALYTICS_URL}/hit",
        json={"site": "fretchen.eu", "path": "/notebook-smoke-test"},
    )
    print(response.status_code)

## 2. Invalid site — expect `400`

Confirms `hit.ts`'s `ALLOWED_SITE` validation is actually live on the target.

In [ ]:
bad_response = requests.post(
    f"{ANALYTICS_URL}/hit",
    json={"site": "evil.com", "path": "/x"},
)
print(bad_response.status_code, bad_response.json())

## 3. Read the write back

Local mode reads the same `notebooks/state/` directory `npm run dev`'s
`FileHitStorage` writes to (`analytics/storage.ts`) — no new plumbing, just the
matching `LocalStorage` class. Live mode reads real S3, confirming the counter is
really there — and that it's **not** publicly readable without these
credentials (see `analytics/README.md` — counters are private by design).

In [ ]:
storage = (
    LocalStorage()
    if LOCAL
    else S3Storage(
        access_key=os.environ["SCW_ACCESS_KEY"],
        secret_key=os.environ["SCW_SECRET_KEY"],
    )
)

hour_key = f"counts/fretchen.eu/{datetime.now(timezone.utc):%Y-%m-%dT%H}.json"
bucket = storage.read(hour_key)
print(hour_key)
print(bucket)

assert bucket is not None, "expected the hour bucket to exist after the POSTs above"
assert bucket["pages"]["/notebook-smoke-test"] == 2